# Model traning logs

In [2]:
import numpy as np
from ultralytics import YOLO
from ultralytics.utils.metrics import DetMetrics
import mlflow
from ultralytics import settings

# Update a setting
settings.update({"mlflow": True})

## Configure mlFlow


In [3]:
import mlflow

# 1. Configure MLflow (Run this once at the top of your notebook)
# This sets the destination for ALL subsequent training runs.
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("inspire-auto-render")

<Experiment: artifact_location='mlflow-artifacts:/1', creation_time=1765292283892, experiment_id='1', last_update_time=1765292283892, lifecycle_stage='active', name='inspire-auto-render', tags={'mlflow.experimentKind': 'custom_model_development'}>

In [4]:
modelv12 = YOLO("yolo12n.pt")

In [5]:
def custom_fitness_v12(self):
    """Custom fitness weights: 80% detection, 20% precision."""
    # [P, R, mAP@0.5, mAP@0.5:0.95]
    w = [0.2, 0.2, 0.2, 0.4]  
    return (np.nan_to_num(np.array(self.mean_results())) * w).sum()

# 2. Apply the patch using property()
DetMetrics.fitness = property(custom_fitness_v12)

In [6]:
# change the fitness function to 0.2 0.2 0.4 0.4
# 
params = {
    "data": "YOLO/data.yaml",
    "task": "detect",
    "mode": "train",
    "epochs": 1,
    "batch": 16,
    "imgsz": 640,
    "patience": 200,
    "hsv_h": 0.1,          # Hue: Full 360 rotation
    "hsv_s": 0.7,          # Saturation: Full range
    "hsv_v": 0.4,          # Value: Full range (Black to White)
    "degrees": 45,        # Rotation: CURRENTLY OFF (See suggestions below)
    "mosaic": 1.0,
    "mixup": 0.2,
    "copy_paste": 0.3,
    "erasing": 0.6,        # Random erasing
    "name": "clean-new-validation-color-with-blur",
}
results = modelv12.train(
    **params
    # data="YOLO/data.yaml",
    # task="detect",
    # mode="train",
    # epochs=250,
    # batch=16,
    # imgsz=640,
    # patience=200,
    # hsv_h=0.1,          # Hue: Full 360 rotation
    # hsv_s=0.7,          # Saturation: Full range
    # hsv_v=0.4,          # Value: Full range (Black to White)
    # degrees=45,        # Rotation: CURRENTLY OFF (See suggestions below)
    # mosaic=1.0,
    # mixup=0.2,
    # copy_paste=0.3,
    # erasing=0.6,        # Random erasing
    # name="clean-new-validation-color-with-blur",
)

New https://pypi.org/project/ultralytics/8.3.235 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.228 🚀 Python-3.12.11 torch-2.9.1 CPU (Apple M3 Pro)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.3, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=YOLO/data.yaml, degrees=45, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=1, erasing=0.6, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.1, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.2, mode=train, model=yolo12n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=clean-new-validation-color-with-blur5, nbs=64, nms=False, opset=None, 

In [7]:
# 4. Export & Log Artifact to the JUST-FINISHED run
# We can get the ID of the run Ultralytics just created
last_run_id = mlflow.last_active_run().info.run_id

with mlflow.start_run(run_id=last_run_id):
    # Export to TFLite
    tflite_path = modelv12.export(format="tflite")
    
    # Upload it to the existing run
    mlflow.log_artifact(tflite_path)
    
    print(f"TFLite model uploaded to run: {last_run_id}")

Ultralytics 8.3.228 🚀 Python-3.12.11 torch-2.9.1 CPU (Apple M3 Pro)
YOLOv12n summary (fused): 159 layers, 2,557,508 parameters, 0 gradients, 6.3 GFLOPs

PyTorch: starting from '/Users/caiofeuser/Developer/inspire/auto_render/runs/detect/clean-new-validation-color-with-blur5/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 8, 8400) (5.2 MB)


2025/12/09 16:37:06 INFO mlflow.tracking.fluent: Autologging successfully enabled for sklearn.
2025/12/09 16:37:06 INFO mlflow.tracking.fluent: Autologging successfully enabled for keras.
2025/12/09 16:37:06 INFO mlflow.tracking.fluent: Autologging successfully enabled for tensorflow.



TensorFlow SavedModel: starting export with tensorflow 2.19.1...

ONNX: starting export with onnx 1.19.1 opset 22...
ONNX: slimming with onnxslim 0.1.75...
ONNX: export success ✅ 2.3s, saved as '/Users/caiofeuser/Developer/inspire/auto_render/runs/detect/clean-new-validation-color-with-blur5/weights/best.onnx' (10.2 MB)
TensorFlow SavedModel: starting TFLite export with onnx2tf 1.28.5...
Saved artifact at '/Users/caiofeuser/Developer/inspire/auto_render/runs/detect/clean-new-validation-color-with-blur5/weights/best_saved_model'. The following endpoints are available:

* Endpoint 'serving_default'
  inputs_0 (POSITIONAL_ONLY): TensorSpec(shape=(1, 640, 640, 3), dtype=tf.float32, name='images')
Output Type:
  TensorSpec(shape=(1, 8, 8400), dtype=tf.float32, name=None)
Captures:
  14495250576: TensorSpec(shape=(4, 2), dtype=tf.int32, name=None)
  14495250384: TensorSpec(shape=(3, 3, 3, 16), dtype=tf.float32, name=None)
  14495250960: TensorSpec(shape=(16,), dtype=tf.float32, name=None)
 

I0000 00:00:1765294639.997244 3474777 devices.cc:76] Number of eligible GPUs (core count >= 8, compute capability >= 0.0): 0 (Note: TensorFlow was not compiled with CUDA or ROCm support)
I0000 00:00:1765294639.997390 3474777 single_machine.cc:374] Starting new session
W0000 00:00:1765294640.585955 3474777 tf_tfl_flatbuffer_helpers.cc:365] Ignored output_format.
W0000 00:00:1765294640.585966 3474777 tf_tfl_flatbuffer_helpers.cc:368] Ignored drop_control_dependency.
I0000 00:00:1765294641.062429 3474777 devices.cc:76] Number of eligible GPUs (core count >= 8, compute capability >= 0.0): 0 (Note: TensorFlow was not compiled with CUDA or ROCm support)
I0000 00:00:1765294641.062488 3474777 single_machine.cc:374] Starting new session
W0000 00:00:1765294641.607151 3474777 tf_tfl_flatbuffer_helpers.cc:365] Ignored output_format.
W0000 00:00:1765294641.607162 3474777 tf_tfl_flatbuffer_helpers.cc:368] Ignored drop_control_dependency.


TensorFlow SavedModel: export success ✅ 50.9s, saved as '/Users/caiofeuser/Developer/inspire/auto_render/runs/detect/clean-new-validation-color-with-blur5/weights/best_saved_model' (25.6 MB)

TensorFlow Lite: starting export with tensorflow 2.19.1...
TensorFlow Lite: export success ✅ 0.0s, saved as '/Users/caiofeuser/Developer/inspire/auto_render/runs/detect/clean-new-validation-color-with-blur5/weights/best_saved_model/best_float32.tflite' (10.1 MB)

Export complete (51.2s)
Results saved to /Users/caiofeuser/Developer/inspire/auto_render/runs/detect/clean-new-validation-color-with-blur5/weights
Predict:         yolo predict task=detect model=/Users/caiofeuser/Developer/inspire/auto_render/runs/detect/clean-new-validation-color-with-blur5/weights/best_saved_model/best_float32.tflite imgsz=640  
Validate:        yolo val task=detect model=/Users/caiofeuser/Developer/inspire/auto_render/runs/detect/clean-new-validation-color-with-blur5/weights/best_saved_model/best_float32.tflite imgsz=6